# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MasoomSakina/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from datasets import load_dataset

print("Loading the data... (Token is already saved!)")
dataset = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train")
df = dataset.to_pandas()

print("\n--- Verifying Unit of Analysis ---")
# Testing the hashed URL identifier
target_col = 'url_hash_id'

if target_col in df.columns:
    is_unique = df[target_col].is_unique
    print(f"Does one row exactly equal one unique URL ({target_col})? {is_unique}")

Loading the data... (Token is already saved!)

--- Verifying Unit of Analysis ---
Does one row exactly equal one unique URL (url_hash_id)? False


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Print out all the columns and their data types
print("--- Dataset Columns and Types ---")
print(df.dtypes)

print("\n--- First 3 Rows ---")
# Display the first few rows to see what the actual data looks like
display(df.head(3))

--- Dataset Columns and Types ---
client_hash_id                    str
content_hash_id                   str
keyword_hash_id                   str
url_hash_id                       str
keyword_char_count              int64
keyword_token_count             int64
url_char_count                  int64
content_created_date           object
content_updated_date           object
content_type                      str
search_volume                 float64
competition                   float64
competition_level                 str
cpc                           float64
main_intent                       str
backlinks                     float64
category_count                  int64
keyword_created_date           object
provider_used                     str
model_used                        str
char_count                    float64
word_count                    float64
last_optimized_date            object
optimization_eligible_date     object
is_published                     bool
is_deleted      

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

print("--- 1. Total Row Count ---")
print(f"Total rows in dataset: {len(df):,}")

print("\n--- 2. Missing Values (Nulls) ---")
# This checks every column and shows you only the ones missing data
missing_data = df.isnull().sum()
if missing_data.sum() == 0:
    print("No missing values found! The data is perfectly clean.")
else:
    print("Warning! These columns have missing data:")
    print(missing_data[missing_data > 0])

print("\n--- 3. Checking the Grain (Unique Values) ---")
# This counts exactly how many unique items exist in every single column
print(df.nunique())

--- 1. Total Row Count ---
Total rows in dataset: 519,606

--- 2. Missing Values (Nulls) ---
Warning! These columns have missing data:
keyword_hash_id                71998
url_hash_id                     6525
search_volume                 142622
competition                   142622
competition_level             144456
cpc                           142622
main_intent                   148398
backlinks                     267474
keyword_created_date           71998
provider_used                 369936
model_used                     84963
char_count                    177768
word_count                    177768
last_optimized_date           474210
optimization_eligible_date    474210
dtype: int64

--- 3. Checking the Grain (Unique Values) ---
client_hash_id                    84
content_hash_id               519606
keyword_hash_id               420094
url_hash_id                   512966
keyword_char_count                88
keyword_token_count               17
url_char_count              

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

print("--- 1. Checking for Unbalanced Clients ---")
# This checks if one client takes up the majority of the dataset
if 'client_hash_id' in df.columns:
    client_counts = df['client_hash_id'].value_counts(normalize=True) * 100
    print("Percentage of data belonging to each client:")
    print(client_counts.round(2).astype(str) + '%')

print("\n--- 2. Checking for Date Limits ---")
# This looks for any time-based columns to see how far back the history goes
date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
if date_cols:
    for col in date_cols:
        # We add .dropna() here to remove the empty 'float' values before calculating!
        clean_dates = df[col].dropna()
        print(f"Date range for {col}: {clean_dates.min()} to {clean_dates.max()}")
else:
    print("No explicit date columns found in this specific table.")

--- 1. Checking for Unbalanced Clients ---
Percentage of data belonging to each client:
client_hash_id
client_78d664d002f13e55    13.55%
client_3ffa76342f366962     6.28%
client_625b6439094e23e4     6.14%
client_73cda7b4e4f265ea     5.96%
client_08a6a72ff48e62c0     5.74%
                            ...  
client_59256b0571e0c970     0.02%
client_a1203ffecad62470     0.01%
client_2b29c545d7e25a1a     0.01%
client_4bc2b415cf2ba0ea      0.0%
client_2c32078d69f2cbad      0.0%
Name: proportion, Length: 84, dtype: str

--- 2. Checking for Date Limits ---
Date range for content_created_date: 2024-10-16 to 2026-07-06
Date range for content_updated_date: 2024-10-28 to 2026-07-06
Date range for keyword_created_date: 2024-10-16 to 2026-07-02
Date range for last_optimized_date: 2026-04-24 to 2026-07-06
Date range for optimization_eligible_date: 2026-06-08 to 2026-08-20


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.